# BQ Partitioning

## Steps

1. Cancel/Drain Harmonization Pipelines (Dataflow)
2. Apply partitioning key-value to FHIR Store configs (Cloudshell)
3. Backup BQ FHIR Datasets (BQ)
4. Parition and Cluster destination tables for Materialized View backup
5. Validate backup operations
6. Drop MVs
7. Purge BQ FHIR Dataset
8. tf-apply-synth (Cloud Build)
9. Cluster newly partitioned BQ Datasets (BQ)
10. Recreate Materialized Views (BQ)

### FHIR Store Config
Patch command to alter configuration of fhir store exports

```sql
curl -X PATCH \
-H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
-H "Content-Type: application/json; charset=utf-8" \
--data '{
        "streamConfigs": [{
                "bigqueryDestination": {
                        "datasetUri": "bq://hcahde040-synth-data.intermediate_fhir",
                        "schemaConfig": {
                                "schemaType": "ANALYTICS_V2",
                                "recursive_structure_depth": 3,
                                "lastUpdatedPartitionConfig" : {"type": "DAY"}
                        }
                }
        }]
}' \
"https://healthcare.googleapis.com/v1beta1/projects/hcahde040-synth-data/locations/us/datasets/healthcare-dataset/fhirStores/intermediate-fhir-store?updateMask=streamConfigs"

```
```sql
curl -X PATCH \
-H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
-H "Content-Type: application/json; charset=utf-8" \
--data '{
        "streamConfigs": [{
                "bigqueryDestination": {
                        "datasetUri": "bq://hcahde040-synth-data.final_fhir",
                        "schemaConfig": {
                                "schemaType": "ANALYTICS_V2",
                                "recursive_structure_depth": 3,
                                "lastUpdatedPartitionConfig" : {"type": "DAY"}
                        }
                }
        }]
}' \
"https://healthcare.googleapis.com/v1beta1/projects/hcahde040-synth-data/locations/us/datasets/healthcare-dataset/fhirStores/final-fhir-store?updateMask=streamConfigs"

```

```sql
curl -X GET -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" "https://healthcare.googleapis.com/v1beta1/projects/hcahde040-synth-data/locations/us/datasets/healthcare-dataset/fhirStores/intermediate-fhir-store"
```

### Partitioning and Clustering MVs to Destination Tables

In [5]:
%%bigquery table_names
SELECT table_name FROM hcahde040-synth-data.clinical_materialized_views.INFORMATION_SCHEMA.TABLES

In [6]:
from google.cloud import bigquery
client = bigquery.Client()
tables = table_names["table_name"].to_list()
tp = bigquery.table.TimePartitioning(field="meta_lastUpdated")
for table in tables:
    table_id = f"hcahde040-synth-data.clinical_materialized_views_history.{table}"
    job_config = bigquery.QueryJobConfig(destination=table_id, clustering_fields=["id"], time_partitioning=tp)
    sql = f"""
    SELECT *, CAST(meta_lastUpdated AS DATE)
    FROM  hcahde040-synth-data.clinical_materialized_views.{table}
    """
    query_job = client.query(sql, job_config=job_config)  # Make an API request.
    query_job.result()  # Wait for the job to complete.
    print("Query results loaded to the table {}".format(table_id))

Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.Procedure
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.Condition
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.testSpecimen
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.Patient
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.Appointment
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.testRelatedPerson
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.Immunization
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.MedicationRequest
Query results loaded to the table hcahde040-stage-data.clinical_materialized_views_history.testPatient
Query results loaded to the table hcahde040-stage-data.clinical_mat

KeyboardInterrupt: 

### Validating Intermediate Fhir Backup

In [4]:
%%bigquery table_names
SELECT table_name FROM hcahde040-synth-data.intermediate_fhir.INFORMATION_SCHEMA.TABLES

In [ ]:
import multiprocessing
from google.cloud import bigquery
client = bigquery.Client()
tables_int = table_names["table_name"].to_list()
tables = [x for x in tables_int if "View" not in x]
def validate(table):
    sql = f"""SELECT COUNT(*) FROM hcahde040-synth-data.intermediate_fhir.{table} as table1 LEFT JOIN hcahde040-synth-data.intermediate_fhir_history.{table} as table2 ON table1.id = table2.id WHERE table2.id IS NULL"""
    query_job = client.query(sql)  # Make an API request.
    print("Number of rows absent in backup of", table, ":", query_job.result().to_dataframe()['f0_'].iloc[0])  # Wait for the job to complete.
    
pool_obj = multiprocessing.Pool()
pool_obj.map(validate, tables)

In [ ]:
%%bigquery table_names
SELECT table_name FROM hcahde040-synth-data.intermediate_fhir.INFORMATION_SCHEMA.TABLES

In [ ]:
import multiprocessing
from google.cloud import bigquery
client = bigquery.Client()
tables_int = table_names["table_name"].to_list()
tables = [x for x in tables_int if "View" not in x]
def validate(table):
    sql = f"""SELECT COUNT(*) FROM hcahde040-synth-data.intermediate_fhir.{table} as table1 LEFT JOIN hcahde040-synth-data.intermediate_fhir_history.{table} as table2 ON table1.id = table2.id WHERE table2.id IS NULL"""
    query_job = client.query(sql)  # Make an API request.
    print("Number of rows absent in backup of", table, ":", query_job.result().to_dataframe()['f0_'].iloc[0])  # Wait for the job to complete.
    
pool_obj = multiprocessing.Pool()
complete = pool_obj.map(validate, tables)

### Validating Final Fhir Backup

In [ ]:
%%bigquery table_names
SELECT table_name FROM hcahde040-synth-data.final_fhir.INFORMATION_SCHEMA.TABLES

In [ ]:
import multiprocessing
from google.cloud import bigquery
client = bigquery.Client()
tables_int = table_names["table_name"].to_list()
tables = [x for x in tables_int if "View" not in x]
def validate(table):
    sql = f"""SELECT COUNT(*) FROM hcahde040-synth-data.final_fhir.{table} as table1 LEFT JOIN hcahde040-synth-data.final_fhir_history.{table} as table2 ON table1.id = table2.id WHERE table2.id IS NULL"""
    query_job = client.query(sql)  # Make an API request.
    print("Number of rows absent in backup of", table, ":", query_job.result().to_dataframe()['f0_'].iloc[0])  # Wait for the job to complete.
    
pool_obj = multiprocessing.Pool()
complete = pool_obj.map(validate, tables)

### Validating MV Backup 

In [1]:
%%bigquery table_names
SELECT table_name FROM hcahde040-synth-data.clinical_materialized_views.INFORMATION_SCHEMA.TABLES

In [2]:
import multiprocessing
from google.cloud import bigquery
client = bigquery.Client()
tables_int = table_names["table_name"].to_list()
tables = [x for x in tables_int if "View" not in x]
def validate(table):
    sql = f"""SELECT COUNT(*) FROM hcahde040-synth-data.clinical_materialized_views.{table} as table1 LEFT JOIN hcahde040-synth-data.clincal_materialized_views_history.{table}_history as table2 ON table1.id = table2.id WHERE table2.id IS NULL"""
    query_job = client.query(sql)  # Make an API request.
    print("Number of rows absent in backup of", table, ":", query_job.result().to_dataframe()['f0_'].iloc[0])  # Wait for the job to complete.
    
pool_obj = multiprocessing.Pool()
complete = pool_obj.map(validate, tables)

MaybeEncodingError: Error sending result: '<multiprocessing.pool.ExceptionWithTraceback object at 0x7f2963dc03d0>'. Reason: 'AttributeError("Can't pickle local object 'if_exception_type.<locals>.if_exception_type_predicate'")'

### Purge BQ and Drop MVs
Commented out because scary

In [28]:
# %%bigquery table_names
# SELECT table_name FROM hcahde040-synth-data.final_fhir.INFORMATION_SCHEMA.TABLES

In [29]:
# for x in tables:
#     !curl -s -X DELETE \
#          -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
#          "https://bigquery.googleapis.com/bigquery/v2/projects/hcahde040-synth-data/datasets/final_fhir/tables/"{x}""    

In [26]:
# %%bigquery table_names
# SELECT table_name FROM hcahde040-synth-data.intermediate_fhir.INFORMATION_SCHEMA.TABLES

In [27]:
# tables = table_names["table_name"].to_list()

# for x in tables:
#     !curl -s -X DELETE \
#          -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
#          "https://bigquery.googleapis.com/bigquery/v2/projects/hcahde040-synth-data/datasets/intermediate_fhir/tables/"{x}"" 

In [24]:
# %%bigquery table_names
# SELECT table_name FROM hcahde040-synth-data.clinical_materialized_views.INFORMATION_SCHEMA.TABLES

In [25]:
# tables = table_names["table_name"].to_list()

# for x in tables:
#     !curl -s -X DELETE \
#          -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
#          "https://bigquery.googleapis.com/bigquery/v2/projects/hcahde040-synth-data/datasets/clinical_materialized_views/tables/"{x}"" 

### Clustering Commands

```BASH
bq update --clustering_fields=id intermediate_fhir.Account
bq update --clustering_fields=id intermediate_fhir.AllergyIntolerance
bq update --clustering_fields=id intermediate_fhir.Appointment
bq update --clustering_fields=id intermediate_fhir.Communication
bq update --clustering_fields=id intermediate_fhir.Condition
bq update --clustering_fields=id intermediate_fhir.Coverage
bq update --clustering_fields=id intermediate_fhir.CoverageEligibilityResponse
bq update --clustering_fields=id intermediate_fhir.Device
bq update --clustering_fields=id intermediate_fhir.DiagnosticReport
bq update --clustering_fields=id intermediate_fhir.DocumentReference
bq update --clustering_fields=id intermediate_fhir.Encounter
bq update --clustering_fields=id intermediate_fhir.Immunization
bq update --clustering_fields=id intermediate_fhir.Location
bq update --clustering_fields=id intermediate_fhir.Medication
bq update --clustering_fields=id intermediate_fhir.MedicationAdministration
bq update --clustering_fields=id intermediate_fhir.MedicationRequest
bq update --clustering_fields=id intermediate_fhir.MessageHeader
bq update --clustering_fields=id intermediate_fhir.Observation
bq update --clustering_fields=id intermediate_fhir.Organization
bq update --clustering_fields=id intermediate_fhir.Patient
bq update --clustering_fields=id intermediate_fhir.Practitioner
bq update --clustering_fields=id intermediate_fhir.Procedure
bq update --clustering_fields=id intermediate_fhir.Provenance
bq update --clustering_fields=id intermediate_fhir.RelatedPerson
bq update --clustering_fields=id intermediate_fhir.ServiceRequest
bq update --clustering_fields=id intermediate_fhir.Specimen
bq update --clustering_fields=id final_fhir.Account
bq update --clustering_fields=id final_fhir.AllergyIntolerance
bq update --clustering_fields=id final_fhir.Appointment
bq update --clustering_fields=id final_fhir.Communication
bq update --clustering_fields=id final_fhir.Condition
bq update --clustering_fields=id final_fhir.Coverage
bq update --clustering_fields=id final_fhir.CoverageEligibilityResponse
bq update --clustering_fields=id final_fhir.Device
bq update --clustering_fields=id final_fhir.DiagnosticReport
bq update --clustering_fields=id final_fhir.DocumentReference
bq update --clustering_fields=id final_fhir.Encounter
bq update --clustering_fields=id final_fhir.Immunization
bq update --clustering_fields=id final_fhir.Location
bq update --clustering_fields=id final_fhir.Medication
bq update --clustering_fields=id final_fhir.MedicationAdministration
bq update --clustering_fields=id final_fhir.MedicationRequest
bq update --clustering_fields=id final_fhir.MessageHeader
bq update --clustering_fields=id final_fhir.Observation
bq update --clustering_fields=id final_fhir.Organization
bq update --clustering_fields=id final_fhir.Patient
bq update --clustering_fields=id final_fhir.Practitioner
bq update --clustering_fields=id final_fhir.Procedure
bq update --clustering_fields=id final_fhir.Provenance
bq update --clustering_fields=id final_fhir.RelatedPerson
bq update --clustering_fields=id final_fhir.ServiceRequest
bq update --clustering_fields=id final_fhir.Specimen
```

### Graveyard

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()
tables = table_names["table_name"].to_list()
# for table in tables:
#     sql = f"""
#     SELECT COUNT(*) FROM hcahde040-synth-data.clinical_materialized_views.{table} as table1
# LEFT JOIN hcahde040-synth-data.testing_mv_copy_time.{table}_history as table2
# ON table1.id = table2.id
# WHERE table2.id IS NULL
#     """
#     query_job = client.query(sql)  # Make an API request.
#     print("Number of rows absent in backup of", table, ":", query_job.result().to_dataframe()['f0_'].iloc[0])  # Wait for the job to complete.